In [1]:
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, Conv1D, MaxPooling1D, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [2]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create output directories for paper artifacts
os.makedirs("report_figures", exist_ok=True)
os.makedirs("saved_models", exist_ok=True)

In [3]:
# ----------------------------------------------------------------------
# 1. DATA INGESTION & CLEANING
# ----------------------------------------------------------------------
DATA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
ZIP_PATH = "jena_climate_2009_2016.csv.zip"
CSV_PATH = "jena_climate_2009_2016.csv"

if not os.path.exists(CSV_PATH):
    print("Downloading Jena Climate dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction complete.")

# Load dataset (sampled every hour: step=6)
raw_df = pd.read_csv(CSV_PATH)
raw_df['Date Time'] = pd.to_datetime(raw_df['Date Time'], format='%d.%m.%Y %H:%M:%S')
raw_df.set_index('Date Time', inplace=True)

# Subsample from 10-minute intervals to 1-hour intervals
df = raw_df.iloc[5::6].copy()

# Fix faulty wind velocity measurements (-9999.0 is an erroneous sensor reading)
bad_wv = df['wv (m/s)'] == -9999.0
df.loc[bad_wv, 'wv (m/s)'] = 0.0
bad_max_wv = df['max. wv (m/s)'] == -9999.0
df.loc[bad_max_wv, 'max. wv (m/s)'] = 0.0

# Feature Engineering: Convert wind direction degrees to vector components (u, v)
wv = df.pop('wv (m/s)')
max_wv = df.pop('max. wv (m/s)')
wd_rad = df.pop('wd (deg)') * np.pi / 180

df['Wx'] = wv * np.cos(wd_rad)
df['Wy'] = wv * np.sin(wd_rad)
df['max Wx'] = max_wv * np.cos(wd_rad)
df['max Wy'] = max_wv * np.sin(wd_rad)

# Time periodicity feature engineering (Diurnal & Annual signals)
timestamp_s = df.index.map(pd.Timestamp.timestamp)
day = 24 * 60 * 60
year = (365.2425) * day

df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))
df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))

TARGET_COL = 'T (degC)'
feature_cols = [col for col in df.columns]

Download and extraction complete.


In [5]:
# ----------------------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS (EDA) ARTIFACTS
# ----------------------------------------------------------------------
print("Generating publication figures for Chapter 4 (EDA)...")

# Figure 1: Atmospheric Variable Correlation Heatmap
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, cmap="coolwarm", annot=False, fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Engineered Climate Variables", fontsize=14, pad=12)
plt.tight_layout()
plt.savefig("report_figures/fig1_correlation_matrix.png", dpi=300)
plt.close()

# Figure 2: Longitudinal Temperature Trend & Seasonality
plt.figure(figsize=(14, 5))
plt.plot(df.index, df[TARGET_COL], color="#1f77b4", linewidth=0.7)
plt.title("Longitudinal Surface Temperature Variations (2009 - 2016)", fontsize=14)
plt.xlabel("Timeline", fontsize=12)
plt.ylabel("Temperature (°C)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("report_figures/fig2_temperature_trend.png", dpi=300)
plt.close()

# Figure 3: Target Distribution & Outlier Detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[TARGET_COL], kde=True, ax=axes[0], color="#2ca02c")
axes[0].set_title("Temperature Probability Density Function")
axes[0].set_xlabel("Temperature (°C)")

sns.boxplot(x=df.index.month, y=df[TARGET_COL], ax=axes[1], palette="tab10")
axes[1].set_title("Monthly Temperature Distribution & Seasonality")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Temperature (°C)")
plt.tight_layout()
plt.savefig("report_figures/fig3_distribution_analysis.png", dpi=300)
plt.close()

Generating publication figures for Chapter 4 (EDA)...


/tmp/ipykernel_809/3780006114.py:32: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x=df.index.month, y=df[TARGET_COL], ax=axes[1], palette="tab10")


In [6]:
# ----------------------------------------------------------------------
# 3. TEMPORAL DATA SPLITTING & SEQUENCE PREPARATION
# ----------------------------------------------------------------------
# Chronological split: 70% Train, 20% Validation, 10% Test
n = len(df)
train_df = df.iloc[0:int(n*0.7)].copy()
val_df = df.iloc[int(n*0.7):int(n*0.9)].copy()
test_df = df.iloc[int(n*0.9):].copy()

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df)
val_scaled = scaler.transform(val_df)
test_scaled = scaler.transform(test_df)

target_idx = df.columns.get_loc(TARGET_COL)

def create_sequences(data, target_index, sequence_length=72, forecast_horizon=24):
    """
    Sequence Generator:
    Takes 72 hours of lookback (3 days) to predict temperature 24 hours into the future.
    """
    X, y = [], []
    for i in range(len(data) - sequence_length - forecast_horizon + 1):
        X.append(data[i:i + sequence_length, :])
        y.append(data[i + sequence_length + forecast_horizon - 1, target_index])
    return np.array(X), np.array(y)

LOOKBACK = 72
HORIZON = 24

X_train, y_train = create_sequences(train_scaled, target_idx, LOOKBACK, HORIZON)
X_val, y_val = create_sequences(val_scaled, target_idx, LOOKBACK, HORIZON)
X_test, y_test = create_sequences(test_scaled, target_idx, LOOKBACK, HORIZON)

# Rescale function to invert standardization for actual °C metrics
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]

def inverse_target(val):
    return (val * target_std) + target_mean

y_train_degC = inverse_target(y_train)
y_val_degC = inverse_target(y_val)
y_test_degC = inverse_target(y_test)

In [8]:
# ----------------------------------------------------------------------
# 4. BENCHMARK BASELINE: RANDOM FOREST REGRESSOR
# ----------------------------------------------------------------------
print("Training Baseline Random Forest Regressor...")
# Flatten sequence window for tabular regressor
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

# Changed n_jobs to None to prevent Colab deadlocks, and estimators to 50
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=None)

# Using [::10] to sample 10% of the data for an even faster baseline
rf_model.fit(X_train_flat[::10], y_train[::10])
rf_pred = inverse_target(rf_model.predict(X_test_flat))
print("Random Forest Training Complete.")

Training Baseline Random Forest Regressor...
Random Forest Training Complete.


In [9]:
# ----------------------------------------------------------------------
# 5. DEEP LEARNING ARCHITECTURES
# ----------------------------------------------------------------------
num_features = X_train.shape[2]

# Architecture 1: Vanilla LSTM
def build_vanilla_lstm():
    model = Sequential([
        Input(shape=(LOOKBACK, num_features)),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Architecture 2: Deep Stacked Bidirectional LSTM
def build_bilstm():
    model = Sequential([
        Input(shape=(LOOKBACK, num_features)),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.2),
        Bidirectional(LSTM(32, return_sequences=False)),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Architecture 3: Hybrid 1D-CNN + BiLSTM (Spatial-Temporal Extractor)
def build_conv_bilstm():
    model = Sequential([
        Input(shape=(LOOKBACK, num_features)),
        Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

models = {
    "Vanilla_LSTM": build_vanilla_lstm(),
    "Stacked_BiLSTM": build_bilstm(),
    "Hybrid_CNN_BiLSTM": build_conv_bilstm()
}

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]

histories = {}
predictions = {"Random Forest (Baseline)": rf_pred}

for name, model in models.items():
    print(f"Training {name}...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=128,
        callbacks=callbacks,
        verbose=1
    )
    histories[name] = history
    model.save(f"saved_models/{name}.keras")

    pred_scaled = model.predict(X_test, batch_size=128).flatten()
    predictions[name] = inverse_target(pred_scaled)

Training Vanilla_LSTM...
Epoch 1/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.1518 - mae: 0.2971 - val_loss: 0.1254 - val_mae: 0.2770 - learning_rate: 0.0010
Epoch 2/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.1191 - mae: 0.2700 - val_loss: 0.1211 - val_mae: 0.2728 - learning_rate: 0.0010
Epoch 3/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - loss: 0.1120 - mae: 0.2621 - val_loss: 0.1227 - val_mae: 0.2760 - learning_rate: 0.0010
Epoch 4/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.1065 - mae: 0.2550 - val_loss: 0.1220 - val_mae: 0.2764 - learning_rate: 0.0010
Epoch 5/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0981 - mae: 0.2449 - val_loss: 0.1269 - val_mae: 0.2804 - learning_rate: 5.0000e-04
Epoch 6/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - loss: 0.0945 - mae: 0.2406 - val_loss: 0.1275 - val_mae: 0.2817 - learning_rate: 5.0000e-04
Epoch 7/20
383/383 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0886 - mae: 0.2329 - val_loss: 0.1276 - val_mae: 0

In [10]:
# ----------------------------------------------------------------------
# 6. EVALUATION METRICS & COMPARATIVE ANALYSIS
# ----------------------------------------------------------------------
results = []
for model_name, preds in predictions.items():
    rmse = np.sqrt(mean_squared_error(y_test_degC, preds))
    mae = mean_absolute_error(y_test_degC, preds)
    r2 = r2_score(y_test_degC, preds)
    results.append({
        "Model Architecture": model_name,
        "RMSE (°C)": round(rmse, 4),
        "MAE (°C)": round(mae, 4),
        "R-squared Score": round(r2, 4)
    })

results_df = pd.DataFrame(results)
results_df.to_csv("report_figures/model_performance_metrics.csv", index=False)
print("\n=== MODEL PERFORMANCE BENCHMARK ===")
print(results_df.to_string(index=False))

# Figure 4: Model Training & Validation Loss Curves
plt.figure(figsize=(10, 6))
for name, hist in histories.items():
    plt.plot(hist.history['loss'], label=f"{name} (Train Loss)", linestyle="--")
    plt.plot(hist.history['val_loss'], label=f"{name} (Val Loss)")
plt.title("Convergence & Loss Trajectory Across Architectures", fontsize=14)
plt.xlabel("Epochs", fontsize=12)
plt.ylabel("Mean Squared Error (Scaled)", fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("report_figures/fig4_loss_curves.png", dpi=300)
plt.close()

# Figure 5: Forecasting vs Actual Comparison (Sample 200 hours)
sample_hours = 200
plt.figure(figsize=(14, 6))
plt.plot(y_test_degC[:sample_hours], label="Ground Truth (Actual °C)", color="black", linewidth=1.5)
plt.plot(predictions["Hybrid_CNN_BiLSTM"][:sample_hours], label="Hybrid CNN-BiLSTM Forecast", color="#d62728", linestyle="--")
plt.plot(predictions["Random Forest (Baseline)"][:sample_hours], label="Random Forest Baseline", color="#1f77b4", alpha=0.7)
plt.title("24-Hour Horizon Forecasting Comparison (200-Hour Window)", fontsize=14)
plt.xlabel("Time Step (Hours)", fontsize=12)
plt.ylabel("Temperature (°C)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("report_figures/fig5_forecast_comparison.png", dpi=300)
plt.close()

# Figure 6: Residual Distribution & Error Variance
plt.figure(figsize=(10, 5))
for name in ["Hybrid_CNN_BiLSTM", "Stacked_BiLSTM", "Random Forest (Baseline)"]:
    residuals = y_test_degC - predictions[name]
    sns.kdeplot(residuals, label=f"{name} Residuals", fill=True, alpha=0.2)
plt.axvline(0, color='grey', linestyle='--')
plt.title("Residual Error Density Distribution", fontsize=14)
plt.xlabel("Forecast Error (°C)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.savefig("report_figures/fig6_residual_distribution.png", dpi=300)
plt.close()

print("\nPipeline execution complete. All high-resolution figures and evaluation tables are saved in /report_figures.")


=== MODEL PERFORMANCE BENCHMARK ===
      Model Architecture  RMSE (°C)  MAE (°C)  R-squared Score
Random Forest (Baseline)     2.8792    2.2364           0.8639
            Vanilla_LSTM     2.6613    2.1046           0.8837
          Stacked_BiLSTM     2.7180    2.1672           0.8787
       Hybrid_CNN_BiLSTM     2.8308    2.2565           0.8684

Pipeline execution complete. All high-resolution figures and evaluation tables are saved in /report_figures.
